# Adding Sports Data Indicators to SOPP Data

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium
from folium.plugins import HeatMap
import datetime

# See all columns
pd.set_option('display.max_columns', None)

In [4]:
# Import Raw Data
## Press "Choose Files" and select "cleanedEaglesData.csv",
## "cleanedFlyersData.csv", "cleaned76ersData.csv", "cleanedPhilliesData.csv",
## and "pa_philadelphia_2020_04_01.csv"
from google.colab import files
files.upload()

rawSOPP = pd.read_csv("pa_philadelphia_2020_04_01.csv", header = 0)
rawNFL = pd.read_csv("cleanedEaglesData.csv", header = 0)
NFL = rawNFL
rawNHL = pd.read_csv("cleanedFlyersData.csv", header = 0)
NHL = rawNHL
rawNBA = pd.read_csv("cleaned76ersData.csv", header = 0)
NBA = rawNBA
rawMLB = pd.read_csv("cleanedPhilliesData.csv", header = 0)
MLB = rawMLB

Saving pa_philadelphia_2020_04_01.csv to pa_philadelphia_2020_04_01.csv


/tmp/ipykernel_309/2759465951.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  rawSOPP = pd.read_csv("pa_philadelphia_2020_04_01.csv", header = 0)


In [ ]:
SOPP = rawSOPP.drop(columns = ['raw_row_number', 'location', 'district',
                               'service_area', 'outcome', 'raw_race',
                               'raw_individual_contraband',
                               'raw_vehicle_contraband'])
SOPP = SOPP.rename(columns={'date': 'Date', 'time': 'StopTime', 'lat': 'Lat',
                            'lng': 'Lng', 'subject_age' : 'Age',
                            'subject_race': 'Race',	'subject_sex': 'Sex',
                            'type': 'StopType', 'arrest_made': 'Arrest',
                            'frisk_performed': 'Frisk',
                            'search_conducted': 'Search',
                            'search_person': 'SearchedPerson',
                            'search_vehicle': 'SearchedVehicle',
                            'contraband_found': 'Contraband'})
SOPP

,Date,StopTime,Lat,Lng,Age,Race,Sex,StopType,Arrest,Contraband,Frisk,Search,SearchedPerson,SearchedVehicle
0,2014-01-01,01:14:00,NaN,NaN,31.0,black,male,pedestrian,True,True,False,True,True,False
1,2014-01-01,01:57:00,NaN,NaN,21.0,black,male,pedestrian,True,False,True,True,True,False
2,2014-01-01,03:30:00,39.950424,-75.192680,24.0,black,male,pedestrian,False,NaN,False,False,False,False
3,2014-01-01,03:40:00,39.950424,-75.192680,20.0,black,male,pedestrian,False,NaN,False,False,False,False
4,2014-01-01,08:30:00,39.983712,-75.234188,31.0,black,male,vehicular,False,NaN,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1865091,2018-04-14,21:36:00,39.928084,-75.221956,60.0,black,male,pedestrian,False,NaN,False,False,False,False
1865092,2018-04-14,22:01:00,39.998242,-75.175190,33.0,asian/pacific islander,male,pedestrian,False,NaN,False,False,False,False
1865093,2018-04-14,22:48:00,40.033812,-75.114429,21.0,black,male,vehicular,True,False,True,True,True,False
1865094,2018-04-14,22:48:00,40.033812,-75.114429,22.0,black,male,vehicular,True,False,True,True,True,False


In [ ]:
# Convert sports dates to date times
SOPP['Date'] = pd.to_datetime(SOPP['Date']).dt.date
NFL['Date'] = pd.to_datetime(NFL['Date']).dt.date
NHL['Date'] = pd.to_datetime(NHL['Date']).dt.date
NBA['Date'] = pd.to_datetime(NBA['Date']).dt.date
MLB['Date'] = pd.to_datetime(MLB['Date']).dt.date

# Reformat Stop Times
SOPP['DateStopTime'] = pd.to_datetime(SOPP['Date'].astype(str) +
                                      ' ' + SOPP['StopTime'].astype(str))
SOPP['StopTimeHour'] = SOPP['DateStopTime'].dt.hour
SOPP = SOPP[['Date', 'DateStopTime', 'Lat', 'Lng', 'Age', 'Race', 'Sex',
             'StopType', 'Arrest', 'Frisk', 'Search', 'SearchedPerson',
             'SearchedVehicle', 'Contraband', 'StopTimeHour']]
SOPP = SOPP.dropna(subset = ['Date', 'DateStopTime', 'Lat', 'Lng'])

# Convert all boolean variables to binary indicators
SOPP['Arrest'] = SOPP['Arrest'].astype(int)
SOPP['Frisk'] = SOPP['Frisk'].astype(int)
SOPP['Search'] = SOPP['Search'].astype(int)
SOPP['SearchedPerson'] = SOPP['SearchedPerson'].astype(int)
SOPP['SearchedVehicle'] = SOPP['SearchedVehicle'].astype(int)
SOPP["Contraband"] = SOPP["Contraband"].astype("Int64")

In [ ]:
SOPP[SOPP['Date'] == datetime.date(2014, 9, 7)]

,Date,DateStopTime,Lat,Lng,Age,Race,Sex,StopType,Arrest,Frisk,Search,SearchedPerson,SearchedVehicle,Contraband,StopTimeHour,EaglesHomeGame
266256,2014-09-07,2014-09-07 00:00:00,40.026767,-75.221397,21.0,white,male,pedestrian,0,0,0,0,0,<NA>,0,0
266257,2014-09-07,2014-09-07 00:00:00,40.027025,-75.142747,25.0,black,female,pedestrian,0,0,0,0,0,<NA>,0,0
266258,2014-09-07,2014-09-07 00:00:00,40.027025,-75.142747,25.0,black,male,pedestrian,0,0,0,0,0,<NA>,0,0
266259,2014-09-07,2014-09-07 00:00:00,40.001047,-75.112250,28.0,hispanic,female,vehicular,0,0,0,0,0,<NA>,0,0
266260,2014-09-07,2014-09-07 00:01:00,39.996231,-75.151839,24.0,white,female,pedestrian,0,0,0,0,0,<NA>,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267404,2014-09-07,2014-09-07 23:54:00,39.905360,-75.174198,23.0,black,male,pedestrian,0,0,0,0,0,<NA>,23,0
267405,2014-09-07,2014-09-07 23:54:00,39.905360,-75.174198,36.0,white,male,pedestrian,0,0,0,0,0,<NA>,23,0
267406,2014-09-07,2014-09-07 23:54:00,39.905360,-75.174198,41.0,white,male,pedestrian,0,0,0,0,0,<NA>,23,0
267407,2014-09-07,2014-09-07 23:55:00,40.042878,-75.016975,31.0,white,male,pedestrian,0,0,0,0,0,<NA>,23,0


In [ ]:
def addGameDayIndicator(gameDates, newColumnName):
  # Adds an indicator column, newColumnName, to SOPP
  # that indicates whether a game took place on the
  # date of each stop

  # Args:
      # gameDates: the unique game dates for a given team
      # newColumnName: the name of the new column

    SOPP[newColumnName] = SOPP['Date'].isin(gameDates).astype(int)

    return SOPP

# Unique Home Game Dates
NFLHomeGames = NFL.loc[NFL['Home'] == 1]['Date'].unique()
NHLHomeGames = NHL.loc[NHL['Home'] == 1]['Date'].unique()
NBAHomeGames = NBA.loc[NBA['Home'] == 1]['Date'].unique()
MLBHomeGames = MLB.loc[MLB['Home'] == 1]['Date'].unique()

In [ ]:
def addGameTimeIndicators(data, startCol, endCol):
  # Adds three indicator columns PreGame, MidGame, and TeamPostGame that
  # indicate when the stop took place in relation to a game

  # Args:
      # SportsData: the sports dataset (NFL, NHL, NBA, or MLB)
      # StartCol: the name of the start time column (StartTime or xStartTime)
      # EndCol: the name of the end time column (EndTime or xEndTime)

    data['PreGame'] = (data['StopTimeHour'] < data[startCol]).astype('Int64')

    data['MidGame'] = ((data['StopTimeHour'] >= data[startCol]) &
     (data['StopTimeHour'] <= data[endCol])).astype('Int64')

    data['PostGame'] = (data['StopTimeHour'] > data[endCol]).astype('Int64')

    return data

In [ ]:
def getDistanceBetween(row, venueCoords):
  # Function to get the distance in miles between two sets of coordinates.

  # Args:
      # row: row object from dataframe; must contain 'Lat' and 'Lng' columns
      # venueCoords: coordinates of event venue

    stopCoords = (row['Lat'], row['Lng'])
    return geodesic(stopCoords, venueCoords).miles

NFLLat = 39.9014
NFLLng = -75.1676

NHLLat = 39.9012
NHLLng = -75.172

NBALat = NHLLat
NBALng = NHLLng

MLBLat = 39.9061
MLBLng = -75.1665

# Eagles

In [ ]:
print(type(SOPP['Date'].iloc[0]))
print(type(NFL['Date'].iloc[0]))

<class 'datetime.date'>
<class 'datetime.date'>


In [ ]:
# Eagles
SOPP = addGameDayIndicator(NFLHomeGames, 'EaglesHomeGame')
SOPPNFL = SOPP[SOPP['EaglesHomeGame'] == 1]
SOPPNFL.drop(columns=['EaglesHomeGame'], inplace=True)

#NFL['StartTime'] = pd.to_datetime(NFL['StartTime'])
#NFL['StartTimeHour'] = NFL['StartTime'].dt.hour

#NFL['xEndTime'] = pd.to_datetime(NFL['xEndTime'])
#NFL['xEndTimeHour'] = NFL['xEndTime'].dt.hour

#SOPPNFL = SOPPNFL.merge(NFL.drop(columns = ['StartTime', 'xEndTime', 'Home']),
                        #on = 'Date', how='left')

#SOPPNFL = addGameTimeIndicators(SOPPNFL, 'StartTimeHour', 'xEndTimeHour')

#SOPPNFL['NFLDistance'] = SOPP.apply(getDistanceBetween, axis = 1, venueCoords = (NFLLat, NFLLng))

## WHY IS THERE NO GAME 1 DATA (09/0702014)

SOPPNFL

/tmp/ipykernel_1382/2123995792.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  SOPPNFL.drop(columns=['EaglesHomeGame'], inplace=True)


,Date,DateStopTime,Lat,Lng,Age,Race,Sex,StopType,Arrest,Frisk,Search,SearchedPerson,SearchedVehicle,Contraband,StopTimeHour
276772,2014-09-15,2014-09-15 00:00:00,39.938690,-75.186919,32.0,black,male,vehicular,0,0,0,0,0,<NA>,0
276773,2014-09-15,2014-09-15 00:00:00,39.989349,-75.131270,24.0,white,female,vehicular,0,0,0,0,0,<NA>,0
276774,2014-09-15,2014-09-15 00:00:00,40.012579,-75.087572,38.0,white,female,pedestrian,0,0,0,0,0,<NA>,0
276775,2014-09-15,2014-09-15 00:00:00,40.048152,-75.069087,24.0,black,male,vehicular,0,0,0,0,0,<NA>,0
276776,2014-09-15,2014-09-15 00:01:00,39.972184,-75.244937,28.0,black,male,vehicular,0,0,0,0,0,<NA>,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1743140,2017-12-17,2017-12-17 23:41:00,39.995880,-75.169186,28.0,black,female,vehicular,0,0,0,0,0,<NA>,23
1743141,2017-12-17,2017-12-17 23:50:00,40.033320,-75.093339,22.0,black,male,vehicular,0,0,0,0,0,<NA>,23
1743142,2017-12-17,2017-12-17 23:50:00,40.037508,-75.179399,23.0,black,female,vehicular,0,0,0,0,0,<NA>,23
1743143,2017-12-17,2017-12-17 23:56:00,40.004672,-75.176977,45.0,black,female,vehicular,0,0,0,0,0,<NA>,23


# Other Sports

In [ ]:
# Flyers
SOPP = addGameDayIndicator(NHLHomeGames, 'FlyersHomeGame')

SOPP

In [ ]:
# 76ers
SOPP = addGameDayIndicator(NBAHomeGames, 'SixersHomeGame')

SOPP

In [ ]:
# Phillies
SOPP = addGameDayIndicator(MLBHomeGames, 'PhilliesHomeGame')

SOPP